# 05 — Customer Segmentation (RFM + K-Means)

**Olist Customer Intelligence Platform**

Two complementary segmentations of the customer base:
1. **RFM quintile scoring** — interpretable, rule-based (this section).
2. **K-Means clustering** — data-driven, added next.

**Modeling note:** ~97% of customers have `frequency = 1`, so Frequency carries
almost no variance. Segmentation is effectively driven by **Recency** and
**Monetary**; we treat frequency as a one-time-vs-repeat signal rather than a
5-way split.

In [1]:
import duckdb
import numpy as np
import pandas as pd
import plotly.express as px
from pathlib import Path

DB_PATH = None
for candidate in (Path("../data/olist.duckdb"), Path("data/olist.duckdb")):
    if candidate.exists():
        DB_PATH = candidate.resolve()
        break
if DB_PATH is None:
    raise FileNotFoundError("olist.duckdb not found — run scripts/load_raw.py and `dbt build`.")

con = duckdb.connect(str(DB_PATH), read_only=True)


def q(sql: str) -> pd.DataFrame:
    """Run SQL against the Olist DuckDB and return a DataFrame."""
    return con.execute(sql).df()


print(f"Connected (read-only) to {DB_PATH.name}")

Connected (read-only) to olist.duckdb


## Section 1 — RFM quintile scoring (your SQL)

Assign each customer an **R, F, M score from 1 to 5** using quintiles, then a
combined label. Return a DataFrame named `rfm` with:

| column | meaning |
|---|---|
| `customer_unique_id` | the customer |
| `recency_days`, `frequency`, `monetary` | the raw RFM values |
| `r_score`, `f_score`, `m_score` | 1–5 quintile scores |
| `rfm_segment` | a business label (see mapping below) |

**Scoring rules — mind the direction:**
- **R score:** *lower* `recency_days` = more recent = **better** = score 5.
  So `ntile(5) over (order by recency_days desc)` (desc, so most-recent gets 5).
- **M score:** *higher* `monetary` = better = 5 → `ntile(5) over (order by monetary asc)`.
- **F score:** because 97% have frequency=1, a clean quintile is impossible.
  Use a simple rule instead, e.g. `least(frequency, 5)` or a CASE:
  `frequency>=3 → 3+`, etc. (Frequency is the weak axis here — keep it simple.)

**Segment mapping (start simple — you can refine):** base it on R and M since
those carry the signal, e.g.
- `Champions`: r_score >= 4 AND m_score >= 4
- `Loyal`: r_score >= 3 AND m_score >= 3 (and not Champions)
- `At Risk`: r_score <= 2 AND m_score >= 3 (spent well, gone quiet)
- `Lost`: r_score <= 2 AND m_score <= 2
- `New/Other`: everything else

Do the scoring in a CTE, then the CASE for the label in the outer query.

In [2]:
rfm = q("""
with base as (
    select
        customer_unique_id,
        recency_days,
        frequency,
        monetary
    from mart_customer_metrics
),

scored as (
    select
        *,
        ntile(5) over (order by recency_days desc) as r_score,
        ntile(5) over (order by monetary asc) as m_score,
        case
            when frequency >= 5 then 5
            when frequency >= 3 then 4
            when frequency >= 2 then 3
            else 1
        end as f_score
    from base
)

select
    customer_unique_id,
    recency_days,
    frequency,
    monetary,
    r_score,
    f_score,
    m_score,
    case
        when r_score >= 4 and m_score >= 4 then 'Champions'
        when r_score >= 3 and m_score >= 3 then 'Loyal'
        when r_score <= 2 and m_score >= 3 then 'At Risk'
        when r_score <= 2 and m_score <= 2 then 'Lost'
        else 'New/Other'
    end as rfm_segment
from scored
""")
rfm["rfm_segment"].value_counts()

rfm_segment
New/Other    21910
At Risk      21910
Loyal        18707
Lost         15434
Champions    15397
Name: count, dtype: int64

### Segment definitions

Labels are driven by **Recency** and **Monetary** scores (1 = worst, 5 = best). Frequency is scored but barely splits this customer base (~97% one-time buyers).

| Segment | Rule | Who they are | What to do |
|---|---|---|---|
| **Champions** | R ≥ 4 and M ≥ 4 | Recent, high spenders | Reward and retain — VIP treatment, early access |
| **Loyal** | R ≥ 3 and M ≥ 3 (not Champions) | Solid recent spenders | Upsell, cross-sell, loyalty perks |
| **At Risk** | R ≤ 2 and M ≥ 3 | Spent well, but haven't ordered in a long time | **Top win-back target** — reactivation emails, offers |
| **Lost** | R ≤ 2 and M ≤ 2 | Low spend, long gone | Low priority; cheap broad campaigns only |
| **New/Other** | Everything else | Recent but low spend, or edge combos | Nurture toward a second purchase |

**Reading tip:** At Risk is often the highest *revenue* segment after Champions — they already proved they'll spend; they just went quiet.

### Segment sizes and value

Once `rfm` is built, this summarizes each segment's size and revenue
contribution (run after your query works).

In [3]:
seg = (rfm.groupby("rfm_segment")
          .agg(customers=("customer_unique_id", "size"),
               avg_recency=("recency_days", "mean"),
               avg_monetary=("monetary", "mean"),
               total_revenue=("monetary", "sum"))
          .reset_index()
          .sort_values("total_revenue", ascending=False))
seg["revenue_share_%"] = (100 * seg["total_revenue"] / seg["total_revenue"].sum()).round(1)
seg

,rfm_segment,customers,avg_recency,avg_monetary,total_revenue,revenue_share_%
0,At Risk,21910,393.035418,242.521269,5313641.01,34.5
1,Champions,15397,90.697149,306.748100,4723000.50,30.6
3,Loyal,18707,167.183300,177.785611,3325835.42,21.6
4,New/Other,21910,133.684162,54.597388,1196228.78,7.8
2,Lost,15434,395.630750,55.790336,861068.04,5.6


**Takeaway:** _Champions (15K customers, 31% of revenue) and At Risk (22K, 35%) are the two money segments — At Risk spent well (avg R$243) but haven't ordered in ~393 days, making them the top win-back target. Lost and New/Other are low-value tails. Frequency barely splits the base (~97% one-time buyers), so R and M drive everything._

## Section 2 — K-Means clustering

Data-driven segmentation: let the algorithm find natural groups in **recency × spend** space, then compare to our rule-based RFM labels from Section 1.

### Step 1 — Prepare features

Before clustering we:

1. **Keep only Recency and Monetary** — frequency is ~constant (97% have 1 order), so it adds noise, not signal.
2. **Log-transform monetary** (`log1p`) — spend is heavily right-skewed; without this, a few whales stretch distances and everyone else collapses into one blob.
3. **Standardize both features** (`StandardScaler`) — rescale to mean 0, std 1 so neither axis dominates Euclidean distance.

The printout below is a sanity check: expect shape `(93358, 2)` and means ≈ `[0, 0]`, stds ≈ `[1, 1]`.

In [4]:
from sklearn.preprocessing import StandardScaler

# Cluster on Recency and Monetary. We deliberately DROP frequency — it's ~constant
# (97% == 1), so it adds noise, not signal. Two features also give a clean 2D plot.
features = rfm[["recency_days", "monetary"]].copy()

# Monetary is heavily right-skewed -> log-transform so a handful of whales
# don't stretch the distance metric and swallow everyone else into one blob.
features["monetary"] = np.log1p(features["monetary"])

# Standardize: mean 0, std 1 on BOTH features, so neither dominates Euclidean distance.
scaler = StandardScaler()
X = scaler.fit_transform(features)

print("shape:", X.shape)
print("means after scaling:", X.mean(axis=0).round(3))  # expect ~[0, 0]
print("stds  after scaling:", X.std(axis=0).round(3))   # expect ~[1, 1]

shape: (93358, 2)
means after scaling: [0. 0.]
stds  after scaling: [1. 1.]


**✓** If means ≈ 0 and stds ≈ 1, scaling worked — proceed to Step 2.

### Step 2 — Choose k (elbow + silhouette)

We don't know the "right" number of clusters upfront, so we try **k = 2 through 8** and compare:

- **Elbow (inertia)** — how tight clusters are internally. Look for the bend where adding another cluster stops helping much.
- **Silhouette score** — how well-separated clusters are (higher = better, max 1). Computed on a 10K sample because full 93K is O(n²) slow.

Pick the k that balances both. We expect modest scores (~0.33–0.36) because most customers are one-time buyers with soft boundaries.

In [6]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import plotly.graph_objects as go
from plotly.subplots import make_subplots

k_range = range(2, 9)
inertias, silhouettes = [], []
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    inertias.append(km.inertia_)
    # silhouette on 93K points is O(n^2) and slow — sample for speed
    silhouettes.append(silhouette_score(X, labels, sample_size=10000, random_state=42))

fig = make_subplots(rows=1, cols=2, subplot_titles=("Elbow (inertia)", "Silhouette score"))
fig.add_trace(go.Scatter(x=list(k_range), y=inertias, mode="lines+markers"), row=1, col=1)
fig.add_trace(go.Scatter(x=list(k_range), y=silhouettes, mode="lines+markers"), row=1, col=2)
fig.update_xaxes(title_text="k")
fig.show()

for k, s in zip(k_range, silhouettes):
    print(f"k={k}: silhouette={s:.3f}")

k=2: silhouette=0.347
k=3: silhouette=0.362
k=4: silhouette=0.346
k=5: silhouette=0.336
k=6: silhouette=0.352
k=7: silhouette=0.340
k=8: silhouette=0.335


**Takeaway:** _Silhouette peaks at **k=3** (0.362) — the best balance of cluster separation in this range. The elbow shows the biggest inertia drop from k=2→3, then diminishing returns after that. Scores stay modest (~0.33–0.36), which fits a customer base that's mostly one-time buyers with soft boundaries, not tight blobs. **Use k=3** for the final K-Means run._

In [14]:
prof = rfm.groupby("cluster").agg(m=("monetary", "mean"), r=("recency_days", "mean"))

champions = prof["m"].idxmax()               # highest average spend
at_risk   = prof["r"].idxmax()               # most days since last order
low_value = (set(prof.index) - {champions, at_risk}).pop()   # the remaining one

cluster_names = {
    champions: "Champions (high-value, active)",
    at_risk:   "At Risk (was valuable, gone quiet)",
    low_value: "Low-Value / New (recent, small spend)",
}
rfm["segment_kmeans"] = rfm["cluster"].map(cluster_names)
rfm["segment_kmeans"].value_counts()

segment_kmeans
Low-Value / New (recent, small spend)    36454
Champions (high-value, active)           29110
At Risk (was valuable, gone quiet)       27794
Name: count, dtype: int64

### Step 4 — Label clusters with business names

K-Means cluster IDs (0, 1, 2) are arbitrary integers — they carry no meaning on their own.

Read the profile table above and map each ID to a business label:
- **High monetary + low recency** → Champions
- **High monetary + high recency** → At Risk (spent well, gone quiet)
- **Low monetary + low recency** → Low-Value / New

Confirm the mapping matches your profile before trusting the labels downstream.

In [15]:
# Cluster IDs are arbitrary integers — you assign business names by READING the profile.
# Confirm against your table: which id has high monetary + low recency = Champions, etc.
cluster_names = {
    1: "Champions (high-value, active)",
    0: "At Risk (was valuable, gone quiet)",
    2: "Low-Value / New (recent, small spend)",
}
rfm["segment_kmeans"] = rfm["cluster"].map(cluster_names)
rfm["segment_kmeans"].value_counts()

segment_kmeans
At Risk (was valuable, gone quiet)       36454
Low-Value / New (recent, small spend)    29110
Champions (high-value, active)           27794
Name: count, dtype: int64

### Step 5 — Visualize segments

Scatter plot of **recency vs monetary** (log y-axis), colored by K-Means segment. We sample 6,000 points for speed — the full 93K would choke the browser.

This should mirror the profile table: Champions top-left (low recency, high spend), At Risk top-right (high recency, moderate spend), Low-Value bottom (low spend).


In [16]:
plot_sample = rfm.sample(6000, random_state=42)
fig = px.scatter(plot_sample, x="recency_days", y="monetary",
                 color="segment_kmeans", opacity=0.5, log_y=True,
                 title="K-Means customer segments (k=3): recency vs monetary")
fig.show()


**Takeaway:** _K-Means finds 3 natural groups: a **high-value core** (~62% of revenue, cluster 2), a **gone-quiet middle** (~22%, cluster 1), and a **low-spend tail** (~17%, cluster 0). This mirrors RFM's Champions / At Risk / New split but discovered from the data rather than hand-cut rules — useful for validating that our RFM labels aren't arbitrary._

### RFM vs K-Means Crosstab

This table compares the two segmentation methods row-by-row:

- **Rows (`segment_kmeans`)** = clusters discovered by K-Means
- **Columns (`rfm_segment`)** = rule-based RFM labels
- **Cells** = number of customers shared by both labels

Use it to check alignment quality:
- A strong diagonal would mean near-perfect agreement
- Off-diagonal mass shows where methods disagree (useful for refining labels)

In this project, the crosstab is mainly a **validation step**: it confirms K-Means and RFM capture a similar high-value vs at-risk vs low-value structure, but with different boundaries.

In [17]:
pd.crosstab(rfm["segment_kmeans"], rfm["rfm_segment"])

rfm_segment,At Risk,Champions,Lost,Loyal,New/Other
segment_kmeans,,,,,
"At Risk (was valuable, gone quiet)",198,227,3583,10536,21910
"Champions (high-value, active)",15889,0,11851,54,0
"Low-Value / New (recent, small spend)",5823,15170,0,8117,0


## Segment Action Plan

| Segment | Customers | Avg spend | Revenue | Action | Why |
|---|---:|---:|---:|---|---|
| Champions | 29,110 | R$326 | R$9.5M (62%) | Retain — VIP perks, loyalty, early access, referrals | Protect the majority of revenue. Avoid broad discounting; they already buy at strong value. |
| At Risk | 27,794 | R$121 | R$3.4M (22%) | Win back — reactivation offer, "we miss you" journey | High historical value but gone quiet (~424 days). Recovering even a small share is material revenue. |
| Low-Value / New | 36,454 | R$70 | R$2.6M (16%) | Grow — onboarding, cross-sell, nudge second purchase | Move first-time/low-spend buyers up the value ladder; second order is the key lever. |

**Execution priority:** `At Risk` (fast ROI) and `Champions` (revenue protection), then `Low-Value / New` (longer-term growth).